# **Big Data MSc Assignment**
## Saul Smith - 22468499

# Hypothesis:

Null Hypothesis ($H_0$): There is no significant difference in consumer reach (mean unique users) between high-rated ($\ge 3$) and low-rated ($< 3$) products.

Alternative Hypothesis ($H_1$): There is a significant difference in consumer reach between high-rated and low-rated products.

Significance Level ($\alpha$): We will use the standard academic threshold of $\alpha = 0.05$.

# Part 1: Amazon Dataset Analysis and Hypothesis Testing

## Environment Setup

In [1]:
# SECTION 1: INITIALISATION & DATA DISCOVERY
from pyspark.sql import SparkSession
import os

# Initialise Spark Session
spark = SparkSession.builder \
    .appName("HPC_BigData_Assignment") \
    .getOrCreate()

print("Spark Session successfully initialised.")

# Automatically find file path in standard Colab environments
possible_paths = [
    "/content/ratings_Electronics.csv",
    "/content/sample_Electronics.csv",
    "/content/sample_data/ratings_Electronics.csv"
]

file_path = None
for path in possible_paths:
    if os.path.exists(path):
        file_path = path
        break

if file_path:
    print(f"Success: Found file at {file_path}")
    raw_df = spark.read.csv(file_path)

    print(f"\nTotal records in raw dataset: {raw_df.count()}")
    print("Sample of raw data (demonstrating missing headers):")
    raw_df.show(5)

    print("Schema before type casting:")
    raw_df.printSchema()
else:
    print("WARNING: ratings_Electronics.csv not found.")

Spark Session successfully initialised.
Success: Found file at /content/ratings_Electronics.csv

Total records in raw dataset: 7824482
Sample of raw data (demonstrating missing headers):
+--------------+----------+---+----------+
|           _c0|       _c1|_c2|       _c3|
+--------------+----------+---+----------+
| AKM1MP6P0OYPR|0132793040|5.0|1365811200|
|A2CX7LUOHB2NDG|0321732944|5.0|1341100800|
|A2NWSAGRHCP8N5|0439886341|1.0|1367193600|
|A2WNBOD3WNDNKT|0439886341|3.0|1374451200|
|A1GI0U4ZRJA8WN|0439886341|1.0|1334707200|
+--------------+----------+---+----------+
only showing top 5 rows
Schema before type casting:
root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)



# Exploratory Data Analysis & Data Cleaning

In [2]:
# SECTION 2: DATA CLEANING & TRANSFORMATION
from pyspark.sql.functions import col, from_unixtime, year, month

# Assigns appropriate column names to the raw headerless file
df = raw_df.toDF("userId", "productId", "rating", "timestamp")

# Cast types and extract temporal components (year and month) for the quarter
df_transformed = df.withColumn("rating", col("rating").cast("float")) \
                   .withColumn("timestamp", col("timestamp").cast("long")) \
                   .withColumn("review_dt", from_unixtime(col("timestamp"))) \
                   .withColumn("year", year(col("review_dt"))) \
                   .withColumn("month", month(col("review_dt")))

print("Transformed/Cleaned Data Schema:")
df_transformed.printSchema()
print("Sample of cleaned data:")
df_transformed.select("userId", "productId", "rating", "review_dt").show(5)

Transformed/Cleaned Data Schema:
root
 |-- userId: string (nullable = true)
 |-- productId: string (nullable = true)
 |-- rating: float (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- review_dt: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)

Sample of cleaned data:
+--------------+----------+------+-------------------+
|        userId| productId|rating|          review_dt|
+--------------+----------+------+-------------------+
| AKM1MP6P0OYPR|0132793040|   5.0|2013-04-13 00:00:00|
|A2CX7LUOHB2NDG|0321732944|   5.0|2012-07-01 00:00:00|
|A2NWSAGRHCP8N5|0439886341|   1.0|2013-04-29 00:00:00|
|A2WNBOD3WNDNKT|0439886341|   3.0|2013-07-22 00:00:00|
|A1GI0U4ZRJA8WN|0439886341|   1.0|2012-04-18 00:00:00|
+--------------+----------+------+-------------------+
only showing top 5 rows


## Section 3: Temporal Filtering

In [3]:
# SECTION 3: TEMPORAL FILTERING
# Isolate records to the specific experimental window (Q2 2014: April, May, June)
q2_2014 = df_transformed.filter((col("year") == 2014) & (col("month") >= 4) & (col("month") <= 6))

print(f"Total records identified for Q2 2014: {q2_2014.count()}")

Total records identified for Q2 2014: 664014


Hypothesis Code:

In [4]:
print("RESEARCH HYPOTHESIS & PARAMETERS")
print("Null Hypothesis (H0): There is no significant difference in consumer reach (mean unique users)")
print("                 between highly-rated products (>= 3) and low-rated products (< 3).")
print("Alternative Hypothesis (H1): There is a significant difference in consumer reach between the groups.")
print("Significance Level (Alpha): 0.05")

RESEARCH HYPOTHESIS & PARAMETERS
Null Hypothesis (H0): There is no significant difference in consumer reach (mean unique users)
                 between highly-rated products (>= 3) and low-rated products (< 3).
Alternative Hypothesis (H1): There is a significant difference in consumer reach between the groups.
Significance Level (Alpha): 0.05


In [5]:
# SECTION 5: ADVANCED METRICS & HYPOTHESIS TESTING
from pyspark.sql.functions import avg, count, countDistinct, stddev
import scipy.stats as stats

# Group by product to create the profiles
product_profiles = q2_2014.groupBy("productId").agg(
    avg("rating").alias("avg_rating"),
    count("userId").alias("review_count"),
    countDistinct("userId").alias("unique_users"),
    stddev("rating").alias("rating_volatility")
)

# Split into high-rated and low-rated cohorts
high_rated_products = product_profiles.filter(col("avg_rating") >= 3)
low_rated_products = product_profiles.filter(col("avg_rating") < 3)

print("METRIC RESULTS ")
print("1. Consumer Reach (Unique Users):")
high_rated_products.select(avg("unique_users")).show()
low_rated_products.select(avg("unique_users")).show()

print("2. Rating Volatility:")
high_rated_products.select(avg("rating_volatility")).show()
low_rated_products.select(avg("rating_volatility")).show()

print("3. Engagement Volume (Review Count):")
high_rated_products.select(avg("review_count")).show()
low_rated_products.select(avg("review_count")).show()

# Perform Welch's T-Test (Independent Two-Sample T-Test)
high_group = high_rated_products.select("unique_users").toPandas()['unique_users']
low_group = low_rated_products.select("unique_users").toPandas()['unique_users']

t_stat, p_val = stats.ttest_ind(high_group, low_group, equal_var=False)

print("\nT-TEST RESULTS")
print(f"T-Statistic: {t_stat:.4f}")
print(f"P-Value:     {p_val:.3e}")

if p_val < 0.05:
    print("DECISION: REJECT THE NULL HYPOTHESIS (H0)")
    print("Conclusion: There is a statistically significant difference in consumer reach.")
else:
    print("DECISION: FAIL TO REJECT THE NULL HYPOTHESIS (H0)")
    print("Conclusion: No statistically significant difference in consumer reach.")

METRIC RESULTS 
1. Consumer Reach (Unique Users):
+-----------------+
|avg(unique_users)|
+-----------------+
|5.978230639664695|
+-----------------+

+-----------------+
|avg(unique_users)|
+-----------------+
|2.690502512562814|
+-----------------+

2. Rating Volatility:
+----------------------+
|avg(rating_volatility)|
+----------------------+
|    1.0177846398080796|
+----------------------+

+----------------------+
|avg(rating_volatility)|
+----------------------+
|    1.3322909377213488|
+----------------------+

3. Engagement Volume (Review Count):
+-----------------+
|avg(review_count)|
+-----------------+
|5.978230639664695|
+-----------------+

+-----------------+
|avg(review_count)|
+-----------------+
|2.690502512562814|
+-----------------+


T-TEST RESULTS
T-Statistic: 27.6865
P-Value:     7.559e-168
DECISION: REJECT THE NULL HYPOTHESIS (H0)
Conclusion: There is a statistically significant difference in consumer reach.


To test these hypotheses, an independent two-sample t-test (Welch’s t-test variant) was conducted. This test does not assume equal population variances between the two groups, making it robust for user engagement distributions which often feature varying dispersion.

Descriptive Statistics

Group 1 (High-Rated Products, $\ge 3$): Mean unique users = $5.98$

Group 2 (Low-Rated Products, $< 3$): Mean unique users = $2.69$


Rating Volatility: High-rated products exhibited a lower mean standard deviation ($\sim 1.02$) compared to low-rated products ($\sim 1.33$), indicating more consistent consumer satisfaction for high-performing items.

 Results and Statistical Findings

 The execution of the Welch's t-test yielded the following output statistics:

 Degrees of Freedom: Approximated using the Welch equation.

 T-Statistic: $27.6865$

 P-Value: $7.559 \times 10^{-168}$

Conclusion:

 P-Value Interpretation: The calculated p-value ($7.559 \times 10^{-168}$) is significantly smaller than the chosen threshold of $\alpha = 0.05$. This provides overwhelming evidence against the null hypothesis.

 Decision: We reject the Null Hypothesis ($H_0$) and accept the Alternative Hypothesis ($H_1$).

 Conclusion: There is a statistically significant difference in consumer reach between high-rated and low-rated products.

# Part 2: MOCKAROO DATA Ingestion

In [6]:
# SECTION 6: END-TO-END ELT PIPELINE SIMULATION
from pyspark.sql.functions import current_timestamp, lit
import os

mock_path = "/content/MOCK_DATA.csv"

if os.path.exists(mock_path):
    # 1. EXTRACT: Read the data from external Kafka-like Mockaroo data source
    raw_stream_df = spark.read.option("header", "true").csv(mock_path)

    # 2. LOAD: Load the extracted dataset directly into HDFS (simulated as Parquet staging)
    hdfs_staged_path = "hdfs_simulated/staged_stream.parquet"
    raw_stream_df.write.mode("overwrite").parquet(hdfs_staged_path)
    print(f"Extraction & Loading completed (Staged at: {hdfs_staged_path})")

    # 3. TRANSFORM: Process and format inside Spark after loading
    loaded_df = spark.read.parquet(hdfs_staged_path)
    transformed_pipeline_df = loaded_df.withColumn("Ingestion_Time", current_timestamp()) \
                                       .withColumn("Source_Topic", lit("amazon_reviews_stream"))

    # Save final transformed dataset
    hdfs_final_path = "hdfs_simulated/processed_stream.parquet"
    transformed_pipeline_df.write.mode("overwrite").parquet(hdfs_final_path)

    print(f"Pipeline Success: Data successfully transformed and stored to {hdfs_final_path}.")
    transformed_pipeline_df.limit(5).show()
else:
    print(f"'{mock_path}' not detected. Upload the Mockaroo CSV to the runtime directory to test the complete pipeline.")

Extraction & Loading completed (Staged at: hdfs_simulated/staged_stream.parquet)
Pipeline Success: Data successfully transformed and stored to hdfs_simulated/processed_stream.parquet.
+---+-----------+----------+--------------------+-----------+--------------+--------------------+--------------------+
| id| first_name| last_name|               email|     gender|    ip_address|      Ingestion_Time|        Source_Topic|
+---+-----------+----------+--------------------+-----------+--------------+--------------------+--------------------+
|  1|     Charin|     Froud| cfroud0@mozilla.com|     Female|   32.28.81.67|2026-05-04 16:07:...|amazon_reviews_st...|
|  2|    Feodora|     Denis|   fdenis1@skype.com|     Female|124.105.166.45|2026-05-04 16:07:...|amazon_reviews_st...|
|  3|     Daphna|Tremethack|dtremethack2@wisc...|     Female| 133.131.75.60|2026-05-04 16:07:...|amazon_reviews_st...|
|  4|Archambault|     Coate|acoate3@mayoclini...|       Male|  57.233.13.66|2026-05-04 16:07:...|amazo

In [7]:
# SECTION 7: STORAGE VERIFICATION
import os

if os.path.exists("hdfs_simulated/processed_stream.parquet"):
    verification_df = spark.read.parquet("hdfs_simulated/processed_stream.parquet")
    print("Successfully verified data recovered from HDFS:")
    verification_df.printSchema()
    verification_df.show(5)
else:
    print("Simulation not run. Verify after running Cell 6.")

Successfully verified data recovered from HDFS:
root
 |-- id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- ip_address: string (nullable = true)
 |-- Ingestion_Time: timestamp (nullable = true)
 |-- Source_Topic: string (nullable = true)

+---+-----------+----------+--------------------+-----------+--------------+--------------------+--------------------+
| id| first_name| last_name|               email|     gender|    ip_address|      Ingestion_Time|        Source_Topic|
+---+-----------+----------+--------------------+-----------+--------------+--------------------+--------------------+
|  1|     Charin|     Froud| cfroud0@mozilla.com|     Female|   32.28.81.67|2026-05-04 16:07:...|amazon_reviews_st...|
|  2|    Feodora|     Denis|   fdenis1@skype.com|     Female|124.105.166.45|2026-05-04 16:07:...|amazon_reviews_st...|
|  3|     Daphna|Tremeth

Analysis:
The architecture simulates a data pipeline. Ingestion is handled by simulating a Kafka stream, which consumes raw, heterogeneous data from edge services or APIs. The data is then processed in memory using Apache Spark for schema conformity, filter invalid records, and add timestamps. Finally, the processed data is persisted into an HDFS-compatible data lake format (Parquet) for long-term distributed storage and analytics.